In [4]:
# ============================================================
# 0) CONFIG (edit these paths / knobs)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/Data/in Use")

ROUNDS_PATH = DATA_DIR / "combined_rounds_all_2017_2026.csv"
ODDS_PATH  = DATA_DIR / "Odds_and_Results.xlsx"         # field for upcoming events
SCHED_2024 = DATA_DIR / "OAD_2024.xlsx"
SCHED_2025 = DATA_DIR / "OAD_2025.xlsx"
SCHED_2026 = DATA_DIR / "OAD_2026.xlsx"

# Modeling years
TRAIN_START_YEAR = 2017
TRAIN_END_YEAR   = 2024   # "driver analysis" and base training window
BACKTEST_YEAR    = 2025

# Pre-event cutoff = start_date - 1 day
CUTOFF_LAG_DAYS = 1

# Rolling windows (rounds)
WINDOWS = (40, 24, 12)

# Stats we will consider (all exist in your screenshot)
# NOTE: We'll create derived birdie_eagle = birdies + eagles_or_better
BASE_STATS = [
    "sg_total", "sg_t2g", "sg_ott", "sg_app", "sg_arg", "sg_putt",
    "round_score",
    "birdies", "eagles_or_better",
    "driving_dist", "driving_acc",
    "gir", "scrambling",
    "prox_fw", "prox_rgh",
    "great_shots", "poor_shots",
    "bogies", "doubles_or_worse", "pars",
]

# Minimum rounds required to trust rolling window values
# (we’ll still keep rows with fewer rounds; these control when rolling values become non-null)
MIN_PERIODS_BY_W = {40: 20, 24: 12, 12: 6}


# ============================================================
# 1) LOAD SCHEDULE (OAD events define the event_id universe)
# ============================================================
def load_sched(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    # expected cols: course_name, course_num, event_id, event_name, purse, rank, start_date, winner_share, year
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["event_id"] = pd.to_numeric(df["event_id"], errors="coerce").astype("Int64")
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
    return df

sched_24 = load_sched(SCHED_2024)
sched_25 = load_sched(SCHED_2025)
sched_26 = load_sched(SCHED_2026)

sched_all = pd.concat([sched_24, sched_25, sched_26], ignore_index=True)
oad_event_ids = sorted(sched_all["event_id"].dropna().astype(int).unique().tolist())

print("OAD event_ids:", len(oad_event_ids), "examples:", oad_event_ids[:10])


# ============================================================
# 2) LOAD ROUNDS (filter to OAD event_ids only)
# ============================================================
use_cols = [
    "year","event_id","event_name","course_num","course_name",
    "round_date","round_num","dg_id","player_name",
    "finish_num","fin_text","event_completed",
] + BASE_STATS

rounds = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in set(use_cols))

# types
rounds["year"] = pd.to_numeric(rounds["year"], errors="coerce").astype("Int64")
rounds["event_id"] = pd.to_numeric(rounds["event_id"], errors="coerce").astype("Int64")
rounds["dg_id"] = pd.to_numeric(rounds["dg_id"], errors="coerce").astype("Int64")
rounds["round_date"] = pd.to_datetime(rounds["round_date"], errors="coerce")

# keep only OAD event universe + usable rows
rounds = rounds[
    rounds["year"].notna()
    & rounds["event_id"].notna()
    & rounds["dg_id"].notna()
    & rounds["round_date"].notna()
].copy()

rounds["event_id"] = rounds["event_id"].astype(int)
rounds["dg_id"] = rounds["dg_id"].astype(int)
rounds["year"] = rounds["year"].astype(int)

rounds = rounds[rounds["event_id"].isin(oad_event_ids)].copy()

# derived stat
if "birdies" in rounds.columns and "eagles_or_better" in rounds.columns:
    rounds["birdie_eagle"] = pd.to_numeric(rounds["birdies"], errors="coerce").fillna(0) + \
                             pd.to_numeric(rounds["eagles_or_better"], errors="coerce").fillna(0)
    if "birdie_eagle" not in BASE_STATS:
        BASE_STATS.append("birdie_eagle")

# ensure numeric stats are numeric
for c in BASE_STATS + ["finish_num"]:
    if c in rounds.columns:
        rounds[c] = pd.to_numeric(rounds[c], errors="coerce")

print("Rounds rows:", len(rounds), "Years:", (rounds["year"].min(), rounds["year"].max()))


# ============================================================
# 3) BUILD PLAYER-EVENT-YEAR TABLE (labels)
#    finish_num is tournament finish (repeated per round); take first non-null per player-event-year
# ============================================================
pe = (
    rounds.groupby(["year","event_id","dg_id"], as_index=False)
    .agg(
        player_name=("player_name","first"),
        course_num=("course_num","first"),
        course_name=("course_name","first"),
        finish_num=("finish_num", lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan),
        fin_text=("fin_text", lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan),
    )
)

# labels (binary)
pe["y_win"]   = (pe["finish_num"] == 1).astype(int)
pe["y_top5"]  = (pe["finish_num"] <= 5).astype(int)
pe["y_top10"] = (pe["finish_num"] <= 10).astype(int)

# keep within modeling years we care about
pe = pe[(pe["year"] >= TRAIN_START_YEAR) & (pe["year"] <= BACKTEST_YEAR)].copy()

print("Player-event rows:", len(pe), "Unique events:", pe[["year","event_id"]].drop_duplicates().shape[0])


# ============================================================
# 4) EVENT START DATES:
#    Prefer schedule start_date for 2024-2026; otherwise infer as min(round_date) per year/event_id
# ============================================================
sched_key = sched_all[["year","event_id","start_date","rank"]].dropna(subset=["year","event_id","start_date"]).copy()
sched_key["year"] = sched_key["year"].astype(int)
sched_key["event_id"] = sched_key["event_id"].astype(int)

# inferred start_date
ev_infer = (
    rounds.groupby(["year","event_id"], as_index=False)
    .agg(infer_start_date=("round_date","min"))
)

ev = pe[["year","event_id"]].drop_duplicates().merge(ev_infer, on=["year","event_id"], how="left")
ev = ev.merge(sched_key, on=["year","event_id"], how="left")

ev["event_start_date"] = ev["start_date"].where(ev["start_date"].notna(), ev["infer_start_date"])
ev["cutoff_date"] = (pd.to_datetime(ev["event_start_date"]) - pd.to_timedelta(CUTOFF_LAG_DAYS, unit="D")).dt.normalize()

# attach cutoff to player-event table
pe = pe.merge(ev[["year","event_id","event_start_date","cutoff_date","rank"]], on=["year","event_id"], how="left")

assert pe["cutoff_date"].notna().mean() > 0.99, "Too many missing cutoff_date; check start_date logic."


# ============================================================
# 5) ROLLING FEATURES PER ROUND (per player), then merge-asof to cutoff_date
#    This avoids per-row slicing and keeps things fast enough.
# ============================================================
roll_stats = [c for c in BASE_STATS if c in rounds.columns]  # only those present

rounds_sorted = rounds.sort_values(["dg_id","round_date"]).copy()

# compute rolling means at each round for each stat/window
for w in WINDOWS:
    minp = MIN_PERIODS_BY_W.get(w, max(3, w//3))
    for stat in roll_stats:
        out_col = f"{stat}_L{w}"
        rounds_sorted[out_col] = (
            rounds_sorted.groupby("dg_id")[stat]
            .rolling(window=w, min_periods=minp)
            .mean()
            .reset_index(level=0, drop=True)
        )

# add "missing split" flags (so missing sg_app etc is not treated as bad performance)
for stat in ["sg_ott","sg_app","sg_arg","sg_putt","sg_t2g"]:
    if stat in rounds_sorted.columns:
        rounds_sorted[f"has_{stat}"] = rounds_sorted[stat].notna().astype(int)

# merge last available rolling snapshot before cutoff_date, per dg_id
feat_cols = [c for c in rounds_sorted.columns if any(c.endswith(f"_L{w}") for w in WINDOWS)]
feat_cols += [c for c in rounds_sorted.columns if c.startswith("has_sg_")]

left = pe.sort_values(["dg_id","cutoff_date"]).copy()
right = rounds_sorted.sort_values(["dg_id","round_date"])[["dg_id","round_date"] + feat_cols].copy()

# --- HARDEN merge_asof inputs ---
left = pe.copy()
right = rounds_sorted[["dg_id", "round_date"] + feat_cols].copy()

# ensure datetimes are comparable + not timezone-aware
left["cutoff_date"] = pd.to_datetime(left["cutoff_date"], errors="coerce").dt.tz_localize(None)
right["round_date"] = pd.to_datetime(right["round_date"], errors="coerce").dt.tz_localize(None)

# enforce int keys
left["dg_id"] = pd.to_numeric(left["dg_id"], errors="coerce").astype("Int64")
right["dg_id"] = pd.to_numeric(right["dg_id"], errors="coerce").astype("Int64")

# drop null keys (merge_asof cannot handle these)
left = left.dropna(subset=["dg_id", "cutoff_date"]).copy()
right = right.dropna(subset=["dg_id", "round_date"]).copy()

left["dg_id"] = left["dg_id"].astype(int)
right["dg_id"] = right["dg_id"].astype(int)

# critical: sort by the "on" key FIRST (global), then the "by" key
left  = left.sort_values(["cutoff_date", "dg_id"], kind="mergesort").reset_index(drop=True)
right = right.sort_values(["round_date", "dg_id"], kind="mergesort").reset_index(drop=True)

X = pd.merge_asof(
    left,
    right,
    by="dg_id",
    left_on="cutoff_date",
    right_on="round_date",
    direction="backward",
    allow_exact_matches=True,
)


# (optional) sanity checks
assert left[["dg_id","cutoff_date"]].isna().sum().sum() == 0
assert right[["dg_id","round_date"]].isna().sum().sum() == 0

X = pd.merge_asof(
    left,
    right,
    by="dg_id",
    left_on="cutoff_date",
    right_on="round_date",
    direction="backward",
    allow_exact_matches=True,
)

X = pd.merge_asof(
    left,
    right,
    by="dg_id",
    left_on="cutoff_date",
    right_on="round_date",
    direction="backward",
    allow_exact_matches=True,
)

# drop helper round_date from merge
X = X.drop(columns=["round_date"], errors="ignore")

print("Feature rows:", len(X), "Feature cols:", len(feat_cols))
print("Non-null rate (sg_total_L40):", X.get("sg_total_L40", pd.Series(dtype=float)).notna().mean())

OAD event_ids: 31 examples: [2, 3, 4, 5, 6, 7, 9, 10, 11, 12]
Rounds rows: 50526 Years: (np.int64(2022), np.int64(2026))
Player-event rows: 16250 Unique events: 123
Feature rows: 16250 Feature cols: 68
Non-null rate (sg_total_L40): 0.7579692307692307


In [5]:
# ============================================================
# 6) EVENT HISTORY FEATURES (known pre-event; no leakage)
#    - prior top10 rate at this event_id
#    - prior mean finish at this event_id
# ============================================================
hist = X[["year","event_id","dg_id","finish_num","y_top10"]].copy()
hist = hist.sort_values(["dg_id","event_id","year"])

# expanding stats, shifted so only PRIOR years are used
hist["prior_top10_rate_event"] = (
    hist.groupby(["dg_id","event_id"])["y_top10"]
    .apply(lambda s: s.expanding().mean().shift(1))
    .reset_index(level=[0,1], drop=True)
)

hist["prior_mean_finish_event"] = (
    hist.groupby(["dg_id","event_id"])["finish_num"]
    .apply(lambda s: s.expanding().mean().shift(1))
    .reset_index(level=[0,1], drop=True)
)

X = X.merge(
    hist[["year","event_id","dg_id","prior_top10_rate_event","prior_mean_finish_event"]],
    on=["year","event_id","dg_id"],
    how="left",
)

# ============================================================
# 7) OPTIONAL: ODDS MERGE (for baseline + maybe later calibration/blending)
#    We will NOT force it into the model unless you choose to later.
# ============================================================
try:
    odds = pd.read_excel(ODDS_PATH)
    # normalize common columns
    for c in ["year","event_id","dg_id"]:
        if c in odds.columns:
            odds[c] = pd.to_numeric(odds[c], errors="coerce")
    if "year" in odds.columns: odds["year"] = odds["year"].astype("Int64")
    if "event_id" in odds.columns: odds["event_id"] = odds["event_id"].astype("Int64")
    if "dg_id" in odds.columns: odds["dg_id"] = odds["dg_id"].astype("Int64")

    # pick an odds column if present
    candidate_odds_cols = ["close_odds","decimal_odds","odds","win_odds","close_odds_decimal"]
    odds_col = next((c for c in candidate_odds_cols if c in odds.columns), None)

    if odds_col:
        odds_small = odds[["year","event_id","dg_id",odds_col]].copy()
        odds_small = odds_small.dropna(subset=["year","event_id","dg_id"])
        odds_small["year"] = odds_small["year"].astype(int)
        odds_small["event_id"] = odds_small["event_id"].astype(int)
        odds_small["dg_id"] = odds_small["dg_id"].astype(int)

        X = X.merge(odds_small, on=["year","event_id","dg_id"], how="left")
        print("Merged odds column:", odds_col, "Non-null rate:", X[odds_col].notna().mean())
    else:
        odds_col = None
        print("No recognized odds column found in Odds_and_Results.xlsx; skipping.")
except Exception as e:
    odds_col = None
    print("Odds merge skipped due to error:", repr(e))


# ============================================================
# 8) DEFINE FEATURE SET (stats-heavy; odds excluded by default)
# ============================================================
id_cols = ["year","event_id","dg_id","player_name","course_num","course_name","event_start_date","cutoff_date","rank","finish_num","fin_text"]
label_cols = ["y_win","y_top5","y_top10"]

# Start with all rolling features + history
feature_cols = [c for c in X.columns if any(c.endswith(f"_L{w}") for w in WINDOWS)]
feature_cols += [c for c in X.columns if c.startswith("has_sg_")]
feature_cols += ["prior_top10_rate_event","prior_mean_finish_event"]

# drop obviously unusable
feature_cols = [c for c in feature_cols if c not in id_cols + label_cols]

# Keep a "sg_total only" baseline feature list for comparisons
sg_total_only = [c for c in feature_cols if c.startswith("sg_total_L")]

print("Full feature count:", len(feature_cols))
print("sg_total-only features:", sg_total_only)


# ============================================================
# 9) MAKE TRAIN/TEST FRAMES (2017-2024 train, 2025 used for walk-forward)
# ============================================================
train_df = X[(X["year"] >= TRAIN_START_YEAR) & (X["year"] <= TRAIN_END_YEAR)].copy()
bt_df    = X[X["year"] == BACKTEST_YEAR].copy()

# sanity
print("Train rows:", len(train_df), "Backtest rows:", len(bt_df))
print("Train positive rates:",
      train_df["y_win"].mean(), train_df["y_top5"].mean(), train_df["y_top10"].mean())


Merged odds column: close_odds Non-null rate: 0.7286769230769231
Full feature count: 70
sg_total-only features: ['sg_total_L40', 'sg_total_L24', 'sg_total_L12']
Train rows: 12380 Backtest rows: 3870
Train positive rates: 0.007431340872374798 0.04426494345718902 0.08707592891760904


In [17]:
# ============================================================
# 10) MODELING: gradient boosting that tolerates NaNs
# ============================================================
from sklearn.experimental import enable_hist_gradient_boosting  # noqa
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42

def fit_hgb(Xtr, ytr):
    # conservative, stable defaults; editable
    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=40,
        l2_regularization=0.0,
        random_state=RANDOM_SEED,
    )
    model.fit(Xtr, ytr)
    return model

def safe_auc(y_true, y_prob):
    # handle cases with no positives in a slice
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def summarize_driver_analysis(df, feat_cols, label_col="y_top10"):
    # simple held-out year: train 2017-2023, validate 2024
    tr = df[df["year"] <= 2023].copy()
    va = df[df["year"] == 2024].copy()

    Xtr, ytr = tr[feat_cols], tr[label_col].astype(int)
    Xva, yva = va[feat_cols], va[label_col].astype(int)

    m = fit_hgb(Xtr, ytr)
    p = m.predict_proba(Xva)[:, 1]

    auc = safe_auc(yva, p)
    brier = brier_score_loss(yva, p) if len(np.unique(yva)) > 1 else np.nan

    # permutation importance on validation (slower but interpretable)
    perm = permutation_importance(
        m, Xva, yva,
        n_repeats=8,
        random_state=RANDOM_SEED,
        scoring="roc_auc" if len(np.unique(yva)) > 1 else None
    )
    imp = pd.DataFrame({
        "feature": feat_cols,
        "perm_importance_mean": perm.importances_mean,
        "perm_importance_std": perm.importances_std,
    }).sort_values("perm_importance_mean", ascending=False)

    return {"model": m, "val_auc": auc, "val_brier": brier, "perm_importance": imp}

# ============================================================
# 11) DRIVER ANALYSIS (run for top10; repeat for top5/win if you want)
# ============================================================
da_top10 = summarize_driver_analysis(train_df, feature_cols, "y_top10")
print("Driver analysis (Top10) - val AUC:", da_top10["val_auc"], "val Brier:", da_top10["val_brier"])
display(da_top10["perm_importance"].head(25))


# ============================================================
# 12) BASELINES: sg_total-only vs full stats-heavy
# ============================================================
def fit_and_eval_simple(df_train, df_test, feat_cols, label_col):
    Xtr = df_train[feat_cols]
    ytr = df_train[label_col].astype(int)
    Xte = df_test[feat_cols]
    yte = df_test[label_col].astype(int)

    m = fit_hgb(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    return {
        "auc": safe_auc(yte, p),
        "brier": brier_score_loss(yte, p) if len(np.unique(yte)) > 1 else np.nan,
        "pred": p,
        "model": m,
    }

# quick static check: train<=2024, test=2025 (not walk-forward yet)
static_res = {}
for lab in ["y_win","y_top5","y_top10"]:
    static_res[(lab,"full")] = fit_and_eval_simple(train_df, bt_df, feature_cols, lab)
    static_res[(lab,"sg_total")] = fit_and_eval_simple(train_df, bt_df, sg_total_only, lab)

for lab in ["y_win","y_top5","y_top10"]:
    print(lab,
          "AUC full:", static_res[(lab,"full")]["auc"],
          "AUC sg_total:", static_res[(lab,"sg_total")]["auc"],
          "Brier full:", static_res[(lab,"full")]["brier"],
          "Brier sg_total:", static_res[(lab,"sg_total")]["brier"])


# ============================================================
# 13) 2025 WALK-FORWARD BACKTEST (event-by-event)
#    Train on all events prior to each 2025 event date, then predict that event.
# ============================================================
# Use schedule order if present; else by inferred event_start_date
sched_2025_key = sched_25[["year","event_id","start_date","rank"]].copy()
sched_2025_key["year"] = sched_2025_key["year"].astype(int)
sched_2025_key["event_id"] = sched_2025_key["event_id"].astype(int)
sched_2025_key["start_date"] = pd.to_datetime(sched_2025_key["start_date"], errors="coerce")

event_order_2025 = (
    bt_df[["event_id","event_start_date"]].drop_duplicates()
    .merge(sched_2025_key[["event_id","start_date","rank"]], on="event_id", how="left")
)
event_order_2025["order_date"] = event_order_2025["start_date"].where(
    event_order_2025["start_date"].notna(), event_order_2025["event_start_date"]
)
event_order_2025 = event_order_2025.sort_values(["order_date","rank","event_id"]).reset_index(drop=True)

def walk_forward_2025(label_col, feat_cols):
    rows = []
    for _, evrow in event_order_2025.iterrows():
        ev_id = int(evrow["event_id"])
        ev_date = pd.to_datetime(evrow["order_date"])
        print(f"\n=== Event {ev_id} | {ev_date.date()} ===")


        # training: all years < 2025 plus 2025 events strictly before this event date
        df_tr = X[
            (X["year"] < BACKTEST_YEAR)
            | ((X["year"] == BACKTEST_YEAR) & (pd.to_datetime(X["event_start_date"]) < ev_date))
        ].copy()

        df_te = X[(X["year"] == BACKTEST_YEAR) & (X["event_id"] == ev_id)].copy()
        if df_te.empty:
            continue

        m = fit_hgb(df_tr[feat_cols], df_tr[label_col].astype(int))
        p = m.predict_proba(df_te[feat_cols])[:, 1]

        df_te = df_te[["year","event_id","dg_id","player_name","finish_num",label_col]].copy()
        df_te["pred"] = p

        # event-level evaluation
        auc = safe_auc(df_te[label_col].astype(int).values, df_te["pred"].values)
        brier = brier_score_loss(df_te[label_col].astype(int).values, df_te["pred"].values) \
                if len(np.unique(df_te[label_col].astype(int).values)) > 1 else np.nan

        # "pick quality": did top predicted player actually achieve the target?
        top_pick = df_te.sort_values("pred", ascending=False).iloc[0]
        rows.append({
            "event_id": ev_id,
            "event_date": ev_date,
            "label": label_col,
            "auc": auc,
            "brier": brier,
            "top_pick_name": top_pick["player_name"],
            "top_pick_dg_id": int(top_pick["dg_id"]),
            "top_pick_pred": float(top_pick["pred"]),
            "top_pick_hit": int(top_pick[label_col]),
            "top_pick_finish": float(top_pick["finish_num"]) if pd.notna(top_pick["finish_num"]) else np.nan,
        })

    return pd.DataFrame(rows).sort_values("event_date")

wf_full = {}
wf_sg   = {}
for lab in ["y_win","y_top5","y_top10"]:
    wf_full[lab] = walk_forward_2025(lab, feature_cols)
    wf_sg[lab]   = walk_forward_2025(lab, sg_total_only)

for lab in ["y_win","y_top5","y_top10"]:
    print("\n====", lab, "====")
    print("Full:   mean hit@1 =", wf_full[lab]["top_pick_hit"].mean(),
          "| mean AUC =", wf_full[lab]["auc"].mean(),
          "| mean Brier =", wf_full[lab]["brier"].mean())
    print("SGonly: mean hit@1 =", wf_sg[lab]["top_pick_hit"].mean(),
          "| mean AUC =", wf_sg[lab]["auc"].mean(),
          "| mean Brier =", wf_sg[lab]["brier"].mean())

# view the per-event picks/results for top10 (most OAD-relevant)
display(wf_full["y_top10"].head(30))


Driver analysis (Top10) - val AUC: 0.7371385417966781 val Brier: 0.07970979459994251


,feature,perm_importance_mean,perm_importance_std
0,sg_total_L40,0.074739,0.005787
42,sg_total_L12,0.024974,0.002169
21,sg_total_L24,0.015065,0.001939
43,sg_t2g_L12,0.010907,0.002657
9,driving_dist_L40,0.010865,0.002174
30,driving_dist_L24,0.010811,0.002874
1,sg_t2g_L40,0.010620,0.002968
4,sg_arg_L40,0.009931,0.005365
69,prior_mean_finish_event,0.009925,0.003479
34,prox_fw_L24,0.008450,0.002298


y_win AUC full: 0.9924963658210724 AUC sg_total: 0.9903872816341621 Brier full: 0.004081104771220021 Brier sg_total: 0.00622631103416534
y_top5 AUC full: 0.9584067172969274 AUC sg_total: 0.8937136118371605 Brier full: 0.031022467939375915 Brier sg_total: 0.040263144535028814
y_top10 AUC full: 0.8995856331168832 AUC sg_total: 0.8574204545454545 Brier full: 0.06026244718817251 Brier sg_total: 0.06679931268731866

=== Event 6 | 2025-01-12 ===

=== Event 2 | 2025-01-19 ===

=== Event 4 | 2025-01-25 ===

=== Event 5 | 2025-02-02 ===

=== Event 3 | 2025-02-09 ===

=== Event 7 | 2025-02-16 ===

=== Event 540 | 2025-02-23 ===

=== Event 10 | 2025-03-02 ===

=== Event 9 | 2025-03-09 ===

=== Event 11 | 2025-03-16 ===

=== Event 475 | 2025-03-23 ===

=== Event 20 | 2025-03-30 ===

=== Event 41 | 2025-04-06 ===

=== Event 14 | 2025-04-13 ===

=== Event 12 | 2025-04-20 ===

=== Event 19 | 2025-05-04 ===

=== Event 480 | 2025-05-11 ===

=== Event 33 | 2025-05-18 ===

=== Event 21 | 2025-05-25 ===



KeyboardInterrupt: 

In [7]:
# # ============================================================
# # ADD-ON: Field-relative features + event/course encodings
# #   - No future info (uses pre-event features + known field only)
# #   - Rolling stats already include ALL tours (your choice)
# # ============================================================
# import numpy as np
# import pandas as pd
# from sklearn.experimental import enable_hist_gradient_boosting  # noqa
# from sklearn.ensemble import HistGradientBoostingClassifier
# from sklearn.metrics import roc_auc_score, brier_score_loss
#
# RANDOM_SEED = 42
#
# # -------------------------
# # EDIT THESE: which base stats to make field-relative
# # Keep small-ish to start. Add more later if it helps.
# # -------------------------
# FIELD_RELATIVE_BASE = [
#     "sg_total_L40", "sg_total_L24", "sg_total_L12",
#     "sg_t2g_L12",
#     "sg_putt_L40",
#     "driving_dist_L40",
#     "round_score_L40",
# ]
#
# # Encodings (optional): which categorical-ish columns
# ENCODE_COLS = ["event_id", "course_num"]  # you can add "rank" later if it’s stable/meaningful
#
# # smoothing strength for target encoding
# TE_M = 50  # higher = more shrink to global prior
#
# def fit_hgb(Xtr, ytr):
#     return HistGradientBoostingClassifier(
#         loss="log_loss",
#         learning_rate=0.05,
#         max_depth=6,
#         max_iter=400,
#         min_samples_leaf=40,
#         l2_regularization=0.0,
#         random_state=RANDOM_SEED,
#     ).fit(Xtr, ytr)
#
# def safe_auc(y_true, y_prob):
#     if len(np.unique(y_true)) < 2:
#         return np.nan
#     return roc_auc_score(y_true, y_prob)
#
# def add_field_relative_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
#     """
#     For each (year,event_id), compute field mean/std of specified cols,
#     then add:
#       - {col}_field_z
#       - {col}_field_mean
#       - {col}_field_std
#     Uses only df values (pre-event features) => no label leakage.
#     """
#     out = df.copy()
#     grp = out.groupby(["year", "event_id"], as_index=False)
#
#     for c in cols:
#         if c not in out.columns:
#             continue
#
#         mean_name = f"{c}_field_mean"
#         std_name  = f"{c}_field_std"
#         z_name    = f"{c}_field_z"
#
#         out[mean_name] = grp[c].transform("mean")
#         out[std_name]  = grp[c].transform("std")
#
#         # avoid div by zero
#         denom = out[std_name].replace(0, np.nan)
#         out[z_name] = (out[c] - out[mean_name]) / denom
#
#     return out
#
# def build_target_encoding_train(df_tr: pd.DataFrame, y_col: str, group_col: str, m: float) -> pd.Series:
#     """
#     Leave-one-out target encoding for TRAIN rows:
#       enc_i = (sum_y_g - y_i + prior*m) / (count_g - 1 + m)
#     This avoids "peeking at its own label" inside training.
#     """
#     prior = df_tr[y_col].mean()
#
#     g = df_tr.groupby(group_col)[y_col].agg(["sum", "count"]).rename(columns={"sum":"sum_y","count":"n"})
#     df = df_tr[[group_col, y_col]].join(g, on=group_col)
#
#     denom = (df["n"] - 1 + m)
#     numer = (df["sum_y"] - df[y_col] + prior * m)
#
#     # handle groups of size 1
#     enc = np.where(df["n"] > 1, numer / denom, prior)
#     return pd.Series(enc, index=df_tr.index)
#
# def build_target_encoding_apply(df_te: pd.DataFrame, df_tr: pd.DataFrame, y_col: str, group_col: str, m: float) -> pd.Series:
#     """
#     Target encoding for TEST rows using TRAIN stats only:
#       enc = (sum_y_g + prior*m) / (count_g + m)
#     Unseen groups -> prior.
#     """
#     prior = df_tr[y_col].mean()
#     g = df_tr.groupby(group_col)[y_col].agg(["sum", "count"]).rename(columns={"sum":"sum_y","count":"n"})
#     df = df_te[[group_col]].join(g, on=group_col)
#     enc = (df["sum_y"] + prior * m) / (df["n"] + m)
#     enc = enc.fillna(prior)
#     return enc
#
# def add_encodings(df_tr: pd.DataFrame, df_te: pd.DataFrame, y_col: str, encode_cols: list[str], m: float):
#     """
#     Adds TE_* columns to both train and test.
#     """
#     tr = df_tr.copy()
#     te = df_te.copy()
#
#     for c in encode_cols:
#         if c not in tr.columns or c not in te.columns:
#             continue
#
#         tr[f"TE_{c}"] = build_target_encoding_train(tr, y_col=y_col, group_col=c, m=m)
#         te[f"TE_{c}"] = build_target_encoding_apply(te, tr, y_col=y_col, group_col=c, m=m)
#
#     return tr, te
#
# # ============================================================
# # 1) Build three feature sets to compare
# # ============================================================
#
# # start from your existing frames
# base_train = train_df.copy()
# base_bt    = bt_df.copy()
#
# # Add field-relative features (safe)
# fr_train = add_field_relative_features(base_train, FIELD_RELATIVE_BASE)
# fr_bt    = add_field_relative_features(base_bt, FIELD_RELATIVE_BASE)
#
# # Helper to collect new field-relative feature columns
# field_rel_cols = []
# for c in FIELD_RELATIVE_BASE:
#     for suf in ["_field_z", "_field_mean", "_field_std"]:
#         cc = f"{c}{suf}"
#         if cc in fr_train.columns:
#             field_rel_cols.append(cc)
#
# print("Field-relative cols added:", len(field_rel_cols))
#
# # Define feature sets (editable)
# FEAT_SG_ONLY              = [c for c in sg_total_only if c in base_train.columns]
# FEAT_SG_ONLY_PLUS_FIELD   = FEAT_SG_ONLY + field_rel_cols
# FEAT_FULL_PLUS_FIELD_BASE = [c for c in feature_cols if c in base_train.columns] + field_rel_cols
#
# # We’ll add TE columns per-label inside walk-forward, since they depend on y_col.
#
# # ============================================================
# # 2) Walk-forward (2025) comparing sets
# # ============================================================
# def walk_forward_2025_compare(
#     X_all: pd.DataFrame,
#     year_bt: int,
#     event_order_df: pd.DataFrame,
#     y_col: str,
#     feat_set_name_to_cols: dict[str, list[str]],
#     use_encodings: bool = False,
#     encode_cols: list[str] = None,
#     te_m: float = 50,
# ):
#     rows = []
#     encode_cols = encode_cols or []
#
#     for _, evrow in event_order_df.iterrows():
#         ev_id = int(evrow["event_id"])
#         ev_date = pd.to_datetime(evrow["order_date"])
#
#         df_tr = X_all[
#             (X_all["year"] < year_bt)
#             | ((X_all["year"] == year_bt) & (pd.to_datetime(X_all["event_start_date"]) < ev_date))
#         ].copy()
#
#         df_te = X_all[(X_all["year"] == year_bt) & (X_all["event_id"] == ev_id)].copy()
#         if df_te.empty or df_tr.empty:
#             continue
#
#         ytr = df_tr[y_col].astype(int)
#         yte = df_te[y_col].astype(int).values
#
#         # For each feature set, fit+predict
#         for name, cols in feat_set_name_to_cols.items():
#             tr_use = df_tr.copy()
#             te_use = df_te.copy()
#
#             # add target encodings if requested (uses TRAIN only; LOO inside train)
#             if use_encodings and encode_cols:
#                 tr_use, te_use = add_encodings(tr_use, te_use, y_col=y_col, encode_cols=encode_cols, m=te_m)
#                 te_cols = [f"TE_{c}" for c in encode_cols if f"TE_{c}" in tr_use.columns]
#                 cols_use = [c for c in cols if c in tr_use.columns] + te_cols
#             else:
#                 cols_use = [c for c in cols if c in tr_use.columns]
#
#             # model
#             m = fit_hgb(tr_use[cols_use], ytr)
#             p = m.predict_proba(te_use[cols_use])[:, 1]
#
#             auc = safe_auc(yte, p)
#             brier = brier_score_loss(yte, p) if len(np.unique(yte)) > 1 else np.nan
#
#             te_tmp = te_use[["year","event_id","dg_id","player_name","finish_num",y_col]].copy()
#             te_tmp["pred"] = p
#             top_pick = te_tmp.sort_values("pred", ascending=False).iloc[0]
#
#             rows.append({
#                 "feature_set": name,
#                 "event_id": ev_id,
#                 "event_date": ev_date,
#                 "label": y_col,
#                 "auc": auc,
#                 "brier": brier,
#                 "top_pick_name": top_pick["player_name"],
#                 "top_pick_dg_id": int(top_pick["dg_id"]),
#                 "top_pick_pred": float(top_pick["pred"]),
#                 "top_pick_hit": int(top_pick[y_col]),
#                 "top_pick_finish": float(top_pick["finish_num"]) if pd.notna(top_pick["finish_num"]) else np.nan,
#             })
#
#     return pd.DataFrame(rows).sort_values(["label","feature_set","event_date"])
#
# # Build event_order_2025 exactly like you already did earlier:
# # event_order_2025 must have columns: event_id, order_date
# # If you already have event_order_2025, skip this section.
# if "event_order_2025" not in globals():
#     sched_2025_key = sched_25[["year","event_id","start_date","rank"]].copy()
#     sched_2025_key["year"] = sched_2025_key["year"].astype(int)
#     sched_2025_key["event_id"] = sched_2025_key["event_id"].astype(int)
#     sched_2025_key["start_date"] = pd.to_datetime(sched_2025_key["start_date"], errors="coerce")
#
#     event_order_2025 = (
#         bt_df[["event_id","event_start_date"]].drop_duplicates()
#         .merge(sched_2025_key[["event_id","start_date","rank"]], on="event_id", how="left")
#     )
#     event_order_2025["order_date"] = event_order_2025["start_date"].where(
#         event_order_2025["start_date"].notna(), event_order_2025["event_start_date"]
#     )
#     event_order_2025 = event_order_2025.sort_values(["order_date","rank","event_id"]).reset_index(drop=True)
#
# # Compare feature sets:
# # - SG-only
# # - SG-only + field-relative
# # - Full + field-relative + target encodings (event/course)
# feat_sets_no_te = {
#     "SG_only": FEAT_SG_ONLY,
#     "SG_only+FieldRel": FEAT_SG_ONLY_PLUS_FIELD,
# }
# feat_sets_full_te = {
#     "Full+FieldRel(+TE)": FEAT_FULL_PLUS_FIELD_BASE,
# }
#
# # For walk-forward we want X_all to include field-relative features.
# # We’ll use fr_all (train+bt combined), computed from X (your master table).
# fr_all = add_field_relative_features(X.copy(), FIELD_RELATIVE_BASE)
#
# results = []
# for y_col in ["y_win","y_top5","y_top10"]:
#     r1 = walk_forward_2025_compare(
#         X_all=fr_all,
#         year_bt=BACKTEST_YEAR,
#         event_order_df=event_order_2025,
#         y_col=y_col,
#         feat_set_name_to_cols=feat_sets_no_te,
#         use_encodings=False,
#     )
#     r2 = walk_forward_2025_compare(
#         X_all=fr_all,
#         year_bt=BACKTEST_YEAR,
#         event_order_df=event_order_2025,
#         y_col=y_col,
#         feat_set_name_to_cols=feat_sets_full_te,
#         use_encodings=True,
#         encode_cols=ENCODE_COLS,
#         te_m=TE_M,
#     )
#     results.append(pd.concat([r1, r2], ignore_index=True))
#
# res = pd.concat(results, ignore_index=True)
#
# # Summary
# summary = (
#     res.groupby(["label","feature_set"], as_index=False)
#     .agg(
#         mean_hit_at1=("top_pick_hit","mean"),
#         mean_auc=("auc","mean"),
#         mean_brier=("brier","mean"),
#         n_events=("event_id","nunique"),
#     )
#     .sort_values(["label","mean_hit_at1","mean_auc"], ascending=[True, False, False])
# )
#
# display(summary)
#
# # Inspect per-event picks for top10 (most relevant)
# display(res[res["label"]=="y_top10"].sort_values(["feature_set","event_date"]).head(60))


Field-relative cols added: 21


,label,feature_set,mean_hit_at1,mean_auc,mean_brier,n_events
2,y_top10,SG_only+FieldRel,0.645161,0.785644,0.078567,31
1,y_top10,SG_only,0.645161,0.785284,0.078908,31
0,y_top10,Full+FieldRel(+TE),0.580645,0.730817,0.185054,31
3,y_top5,Full+FieldRel(+TE),0.451613,0.761765,0.092072,31
4,y_top5,SG_only,0.387097,0.787262,0.047898,31
5,y_top5,SG_only+FieldRel,0.354839,0.799698,0.047763,31
7,y_win,SG_only,0.129032,0.856306,0.008357,31
8,y_win,SG_only+FieldRel,0.096774,0.847356,0.008605,31
6,y_win,Full+FieldRel(+TE),0.096774,0.797884,0.015768,31


,feature_set,event_id,event_date,label,auc,brier,top_pick_name,top_pick_dg_id,top_pick_pred,top_pick_hit,top_pick_finish
248,Full+FieldRel(+TE),6,2025-01-12,y_top10,0.691406,0.103103,"Spaun, J.J.",17536,0.039160,1,3.0
249,Full+FieldRel(+TE),2,2025-01-19,y_top10,0.452740,0.258869,"Spaun, J.J.",17536,0.993616,0,29.0
250,Full+FieldRel(+TE),4,2025-01-25,y_top10,0.575510,0.089432,"Manassero, Matteo",11460,0.259404,0,25.0
251,Full+FieldRel(+TE),5,2025-02-02,y_top10,0.728590,0.136510,"Davis, Cam",17786,0.008459,1,5.0
252,Full+FieldRel(+TE),3,2025-02-09,y_top10,0.637115,0.083248,"Thomas, Justin",14139,0.001385,1,6.0
253,Full+FieldRel(+TE),7,2025-02-16,y_top10,0.523100,0.135402,"Morikawa, Collin",22085,0.446713,0,17.0
254,Full+FieldRel(+TE),540,2025-02-23,y_top10,0.835417,0.090624,"Bhatia, Akshay",26096,0.003676,1,9.0
255,Full+FieldRel(+TE),10,2025-03-02,y_top10,0.767910,0.069036,"Clanton, Luke",32330,0.008542,0,18.0
256,Full+FieldRel(+TE),9,2025-03-09,y_top10,0.645161,0.138753,"McIlroy, Rory",10091,0.001446,0,15.0
257,Full+FieldRel(+TE),11,2025-03-16,y_top10,0.825701,0.075421,"McIlroy, Rory",10091,0.014830,1,1.0


In [21]:
# ============================================================
# WIN-ONLY MODEL (P(win)) — 2025 walk-forward, NO FUTURE INFO
# Assumes you already built X with:
#   columns: year, event_id, event_start_date, player_name, dg_id, finish_num, y_win
#   and rolling features like sg_total_L40/L24/L12
# ============================================================
import numpy as np
import pandas as pd

from sklearn.experimental import enable_hist_gradient_boosting  # no
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score

# -----------------------------
# EDIT: feature set (SG-only)
# -----------------------------
WIN_FEATURES = ["sg_total_L40", "sg_total_L24", "sg_total_L12"]

# Optional: add simple recency shape features (often helps a bit)
# Uncomment if you want:
# WIN_FEATURES += ["sg_total_L12_minus_L40"]
# (we'll build it below if used)

# -----------------------------
# Calibration
# -----------------------------
USE_CALIBRATION = True
CAL_METHOD = "sigmoid"   # "sigmoid" (safer) or "isotonic" (can overfit)
CAL_FOLDS = 5            # grouped by year within training window

RANDOM_SEED = 42

def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def build_base_model():
    return HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=40,
        l2_regularization=0.0,
        random_state=RANDOM_SEED,
    )

# Build event_name lookup from OAD schedule (2025)
sched = pd.read_excel(SCHED_2025)  # or whatever your 2025 schedule path variable is
sched["event_id"] = pd.to_numeric(sched["event_id"], errors="coerce").astype("Int64")

event_name_map = (
    sched.dropna(subset=["event_id"])
         .drop_duplicates(subset=["event_id"])
         .set_index("event_id")["event_name"]
         .to_dict()
)

def fit_model_with_year_sigmoid_calibration(train_df, feat_cols, label_col="y_win"):
    """
    Time-safe calibration:
      1) make OOF base-model probabilities using year-based folds
      2) fit sigmoid calibrator (logistic regression) on OOF probs
      3) refit base model on full train
    Returns: (base_model, calibrator_or_None)
    """
    tr = train_df.dropna(subset=feat_cols + [label_col, "year"]).copy()
    Xtr = tr[feat_cols]
    ytr = tr[label_col].astype(int).values
    years = tr["year"].astype(int).values
    uniq_years = np.unique(years)

    base = build_base_model()

    # If not enough distinct years, skip calibration
    if (not USE_CALIBRATION) or (len(uniq_years) < 3):
        base.fit(Xtr, ytr)
        return base, None

    # Build year folds: each fold holds out one year (or a few years if many)
    # Simple LOYO:
    oof = np.zeros(len(tr), dtype=float)
    oof[:] = np.nan

    for y in uniq_years:
        idx_te = np.where(years == y)[0]
        idx_tr = np.where(years != y)[0]

        # Need both classes in training fold
        if len(np.unique(ytr[idx_tr])) < 2:
            continue

        m = build_base_model()
        m.fit(Xtr.iloc[idx_tr], ytr[idx_tr])
        oof[idx_te] = m.predict_proba(Xtr.iloc[idx_te])[:, 1]

    ok = np.isfinite(oof)
    if ok.mean() < 0.8 or len(np.unique(ytr[ok])) < 2:
        # too many NaNs or degenerate folds -> skip calibration
        base.fit(Xtr, ytr)
        return base, None

    # Sigmoid calibration: P = sigmoid(a * logit(p) + b)
    # To stabilize, calibrate on logit(p) with clipping
    eps = 1e-6
    p = np.clip(oof[ok], eps, 1 - eps)
    logit_p = np.log(p / (1 - p)).reshape(-1, 1)

    cal = LogisticRegression(solver="lbfgs")
    cal.fit(logit_p, ytr[ok])

    # Refit base on full training
    base.fit(Xtr, ytr)
    return base, cal

def predict_proba_calibrated(base_model, calibrator, X):
    p = base_model.predict_proba(X)[:, 1]
    if calibrator is None:
        return p
    eps = 1e-6
    p2 = np.clip(p, eps, 1 - eps)
    logit_p = np.log(p2 / (1 - p2)).reshape(-1, 1)
    return calibrator.predict_proba(logit_p)[:, 1]


# ============================================================
# 0) Prepare X and event order for 2025
# ============================================================
BACKTEST_YEAR = 2025

df_all = X.copy()

# Create optional derived features if requested
if "sg_total_L12_minus_L40" in WIN_FEATURES and "sg_total_L12_minus_L40" not in df_all.columns:
    df_all["sg_total_L12_minus_L40"] = df_all["sg_total_L12"] - df_all["sg_total_L40"]

# Ensure event_start_date exists and is datetime
df_all["event_start_date"] = pd.to_datetime(df_all["event_start_date"], errors="coerce")

# Build event order for 2025
bt_events = (
    df_all[df_all["year"] == BACKTEST_YEAR][["event_id","event_start_date"]]
    .dropna()
    .drop_duplicates()
    .sort_values("event_start_date")
    .reset_index(drop=True)
)

# If you have rank/order in schedule and want it, merge it here (optional)
# bt_events = bt_events.merge(sched_25[["event_id","rank"]], on="event_id", how="left").sort_values(["event_start_date","rank","event_id"])

print("2025 events found:", len(bt_events))

# ============================================================
# 1) Walk-forward: for each 2025 event, train on ALL prior data, predict that event
# ============================================================
event_boards = []   # per-player per-event predictions
event_summ   = []   # per-event summary
event_top10 = []

# ============================
# Add: event_skill + odds blend (stats-first)
# ============================

# 1) Load once, ABOVE the loop (do this near where you set BACKTEST_YEAR)
EVENT_SKILL_PATH = "/Data/in Use/event_skill.xlsx"
event_skill_df = pd.read_excel(EVENT_SKILL_PATH)
event_skill_df["year"] = pd.to_numeric(event_skill_df["year"], errors="coerce").astype("Int64")
event_skill_df["event_id"] = pd.to_numeric(event_skill_df["event_id"], errors="coerce").astype("Int64")

# Optional: keep just what we need
event_skill_df = event_skill_df[["year","event_id","event_name","avg_skill","x_score"]].copy()

def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def _sigmoid(z):
    return 1 / (1 + np.exp(-z))

def blend_stats_odds_with_field_strength(board, year, ev_id, event_skill_df):
    """
    board must contain:
      - p_win_raw  (stats model prob, NOT normalized)
      - close_odds (decimal odds)
    returns board with:
      - p_win_blend_raw
      - p_win_blend_norm
      - blend_weight
      - x_score (if found)
    """
    b = board.copy()

    # Pull event x_score (pre-event, safe)
    row = event_skill_df[(event_skill_df["year"] == year) & (event_skill_df["event_id"] == ev_id)]
    if row.empty:
        x = np.nan
        # conservative default if missing
        w_event = 0.20
        ev_name = None
    else:
        x = float(row["x_score"].iloc[0])
        ev_name = row["event_name"].iloc[0] if "event_name" in row.columns else None

        # Map x_score -> weight. Clamp to keep odds "a little bit".
        # You can tune these later, but start simple:
        # typical x_score seems around ~0.7–1.0 from your image
        w_event = 0.10 + 0.30 * ((x - 0.70) / (1.00 - 0.70))  # 0.10..0.40 roughly
        w_event = float(np.clip(w_event, 0.10, 0.35))

    # Odds implied prob within event
    p_odds_raw = 1.0 / pd.to_numeric(b["close_odds"], errors="coerce")
    p_odds_raw = p_odds_raw.fillna(p_odds_raw.median())
    p_odds = p_odds_raw / (p_odds_raw.sum() + 1e-12)

    # Stats prob: use p_win_raw (not normalized) and normalize to an event prior first
    p_stats_raw = pd.to_numeric(b["p_win_raw"], errors="coerce").fillna(0.0)
    p_stats = p_stats_raw / (p_stats_raw.sum() + 1e-12)

    # Blend in logit space, then renormalize
    z = _logit(p_stats.values) + w_event * _logit(p_odds.values)
    p_blend = _sigmoid(z)

    b["p_win_blend_raw"] = p_blend
    b["p_win_blend_norm"] = b["p_win_blend_raw"] / (b["p_win_blend_raw"].sum() + 1e-12)
    b["blend_weight"] = w_event
    b["x_score"] = x
    if ev_name is not None:
        b["event_name"] = ev_name
    return b


for _, ev in bt_events.iterrows():
    ev_id = int(ev["event_id"])
    ev_date = pd.to_datetime(ev["event_start_date"])
    ev_name = sched.loc[sched["event_id"] == ev_id, "event_name"].iloc[0] if (sched is not None and (sched["event_id"] == ev_id).any()) else ""
    ev_name = event_name_map.get(ev_id, "UNKNOWN_EVENT")
    print(f"\n=== Event {ev_id} | {ev_date.date()} | {ev_name} ===")

    # Training data = all years < 2025 plus 2025 events strictly before this event date
    train_df = df_all[
        (df_all["year"] < BACKTEST_YEAR) |
        ((df_all["year"] == BACKTEST_YEAR) & (df_all["event_start_date"] < ev_date))
    ].copy()

    test_df = df_all[(df_all["year"] == BACKTEST_YEAR) & (df_all["event_id"] == ev_id)].copy()
    if test_df.empty or train_df.empty:
        continue

    # require features present
    feat_cols = [c for c in WIN_FEATURES if c in train_df.columns]
    missing = set(WIN_FEATURES) - set(feat_cols)
    if missing:
        print("Missing features for event", ev_id, ":", sorted(missing))
        continue

    base_model, calibrator = fit_model_with_year_sigmoid_calibration(train_df, feat_cols, label_col="y_win")
    p = predict_proba_calibrated(base_model, calibrator, test_df[feat_cols])
    was_cal = int(calibrator is not None)

    test_df = test_df.copy()
    test_df["p_win"] = p
    test_df["was_calibrated"] = int(was_cal)

    # Mark actual winner (finish_num == 1)
    test_df["is_winner"] = (pd.to_numeric(test_df["finish_num"], errors="coerce") == 1).astype(int)

    # Board
    board_cols = ["year","event_id","event_start_date","dg_id","player_name","finish_num","is_winner","p_win","was_calibrated"] + feat_cols
    board = test_df[board_cols].copy()
    board = board.sort_values("p_win", ascending=False).reset_index(drop=True)
    board["pred_rank"] = np.arange(1, len(board)+1)
    event_boards.append(board)

        # Normalize win probs to sum to ~1 per event (win-share)
    eps = 1e-12
    board["p_win_raw"] = board["p_win"]
    board["p_win_norm"] = board["p_win_raw"] / (board["p_win_raw"].sum() + eps)

        # Ensure close_odds is on board (bring it from test_df if needed)
    if "close_odds" not in board.columns and "close_odds" in test_df.columns:
        board = board.merge(
            test_df[["dg_id","close_odds"]],
            on="dg_id",
            how="left"
        )

    board = blend_stats_odds_with_field_strength(
        board=board,
        year=BACKTEST_YEAR,
        ev_id=ev_id,
        event_skill_df=event_skill_df
    )

    # If you want to rank/display by blended instead of stats-only:
    board = board.sort_values("p_win_blend_norm", ascending=False).reset_index(drop=True)
    board["pred_rank"] = np.arange(1, len(board)+1)


    top10_board = board.head(10)[[
        "year","event_id","event_start_date","pred_rank",
        "player_name","dg_id",
        "p_win_norm","p_win_raw",
        "sg_total_L40","sg_total_L24","sg_total_L12",
    ]].copy()

    # nice formatting for display
    top10_board["p_win_norm_pct"] = (100 * top10_board["p_win_norm"]).round(2)
    top10_board["p_win_raw_pct"]  = (100 * top10_board["p_win_raw"]).round(2)

    display(
        top10_board[[
            "pred_rank","player_name","dg_id",
            "p_win_norm_pct","p_win_raw_pct",
            "sg_total_L40","sg_total_L24","sg_total_L12"
        ]]
    )


    # Event summary: top pick + winner rank
    top_pick = board.iloc[0]
    winner_rows = board[board["is_winner"] == 1]
    winner_rank = int(winner_rows["pred_rank"].iloc[0]) if not winner_rows.empty else np.nan
    winner_name = winner_rows["player_name"].iloc[0] if not winner_rows.empty else None
    winner_prob = float(winner_rows["p_win_norm"].iloc[0]) if not winner_rows.empty else np.nan

    eval_board = board.dropna(subset=["finish_num"]).copy()
    y_true = eval_board["is_winner"].values
    y_prob = eval_board["p_win_norm"].values
    brier = brier_score_loss(y_true, y_prob)
    auc = safe_auc(y_true, y_prob)


    event_summ.append({
        "event_id": ev_id,
        "event_date": ev_date,
        "calibrated": int(was_cal),
        "top_pick": top_pick["player_name"],
        "top_pick_pwin": float(top_pick["p_win_norm"]),
        "top_pick_won": int(top_pick["is_winner"]),
        "winner_name": winner_name,
        "winner_rank": winner_rank,
        "winner_pwin": winner_prob,
        "brier": brier,
        "auc": auc,
        "n_field": len(board),
    })

boards_2025 = pd.concat(event_boards, ignore_index=True) if event_boards else pd.DataFrame()
summ_2025   = pd.DataFrame(event_summ).sort_values("event_date") if event_summ else pd.DataFrame()
top10_all_2025 = pd.concat(event_top10, ignore_index=True) if event_top10 else pd.DataFrame()


print("Events scored:", summ_2025["event_id"].nunique() if not summ_2025.empty else 0)
display(summ_2025.head(40))

# ============================================================
# 2) "What % of the time these players won" — in a meaningful way
#    A) Per-event: show the board with P(win) and actual winner
#    B) Calibration: do predicted probabilities match observed win rates (aggregate)?
#    C) Per-player in 2025: avg P(win) vs actual wins
# ============================================================

# A) Example: show top 15 for a chosen event_id (edit)
EXAMPLE_EVENT_ID = int(summ_2025["event_id"].iloc[0]) if not summ_2025.empty else None
if EXAMPLE_EVENT_ID is not None:
    display(
        boards_2025[boards_2025["event_id"] == EXAMPLE_EVENT_ID]
        .sort_values("p_win", ascending=False)
        .head(15)
    )

# B) Calibration table (aggregate across all 2025 player-event rows)
# B) Calibration table (aggregate across all 2025 player-event rows) — use p_win_norm
if not boards_2025.empty:
    tmp = boards_2025.copy()

    # If you want calibration only on players with known outcomes:
    # tmp = tmp.dropna(subset=["finish_num"]).copy()

    bins = [0, 0.002, 0.005, 0.01, 0.02, 0.04, 0.07, 0.10, 0.20, 1.0]
    tmp["p_bin"] = pd.cut(tmp["p_win_norm"], bins=bins, include_lowest=True)

    calib = (
        tmp.groupby("p_bin", as_index=False)
        .agg(
            n=("p_win_norm","size"),
            avg_pred_pwin=("p_win_norm","mean"),
            observed_win_rate=("is_winner","mean"),
        )
    )
    display(calib)

# C) Per-player summary (2025): "model says you win X% on average when you tee it up"
if not boards_2025.empty:
    player_2025 = (
        boards_2025.groupby(["dg_id","player_name"], as_index=False)
        .agg(
            starts=("event_id","nunique"),
            wins=("is_winner","sum"),
            avg_pwin=("p_win","mean"),
            max_pwin=("p_win","max"),
        )
        .sort_values(["wins","avg_pwin","starts"], ascending=[False, False, False])
    )
    display(player_2025.head(50))

2025 events found: 31

=== Event 6 | 2025-01-12 | Sony Open in Hawaii ===


,pred_rank,player_name,dg_id,p_win_norm_pct,p_win_raw_pct,sg_total_L40,sg_total_L24,sg_total_L12
0,1,"Henley, Russell",14578,3.84,5.45,1.302175,1.322500,1.076083
1,2,"Matsuyama, Hideki",13562,2.39,3.39,1.580925,1.627917,1.633583
2,3,"McNealy, Maverick",18634,2.96,4.20,1.117150,0.975458,1.002417
3,4,"Clanton, Luke",32330,2.63,3.73,1.660250,1.660250,1.809500
4,5,"Stevens, Sam",25569,3.22,4.58,0.463625,0.631708,-0.054167
5,6,"Horschel, Billy",11276,2.44,3.46,1.160025,1.018750,0.587833
6,7,"Phillips, Chandler",27164,2.75,3.90,0.062000,0.265250,-0.800000
7,8,"Hoey, Rico",23504,2.56,3.64,0.573400,0.778958,0.355167
8,9,"Novak, Andrew",23475,1.95,2.76,0.737125,0.735833,0.055083
9,10,"McCarthy, Denny",19870,1.77,2.52,0.555325,0.663792,1.260583



=== Event 2 | 2025-01-19 | The American Express ===


,pred_rank,player_name,dg_id,p_win_norm_pct,p_win_raw_pct,sg_total_L40,sg_total_L24,sg_total_L12
0,1,"Meissner, Mac",28159,8.49,10.98,0.897600,0.632375,0.765000
1,2,"Greyserman, Max",23465,3.16,4.09,0.921875,1.453292,0.469167
2,3,"Hall, Harry",27194,2.56,3.31,0.793675,1.302875,1.564917
3,4,"Spaun, J.J.",17536,2.28,2.95,1.009425,2.055917,2.287333
4,5,"Smalley, Alex",18474,2.22,2.87,-0.479150,0.251500,1.336917
5,6,"Johnson, Zach",6986,2.49,3.22,0.253475,0.310750,0.370667
6,7,"Cantlay, Patrick",15466,1.22,1.58,1.320875,1.771667,1.368000
7,8,"Thomas, Justin",14139,1.08,1.39,0.790000,0.620542,0.034667
8,9,"Taylor, Nick",13126,1.90,2.45,-0.004425,0.263458,0.378333
9,10,"Burns, Sam",19483,1.15,1.48,1.267275,1.202792,1.510917



=== Event 4 | 2025-01-25 | Farmers Insurance Open ===


...

Events scored: 31


,event_id,event_date,calibrated,top_pick,top_pick_pwin,top_pick_won,winner_name,winner_rank,winner_pwin,brier,auc,n_field
0,6,2025-01-12,1,"Henley, Russell",0.038401,0,"Taylor, Nick",111,0.001445,0.013240,0.106667,143
1,2,2025-01-19,1,"Meissner, Mac",0.084884,0,"Straka, Sepp",18,0.012706,0.013929,0.814286,156
2,4,2025-01-25,1,"Griffin, Ben",0.036481,0,"English, Harris",13,0.015676,0.013918,0.927536,154
3,5,2025-02-02,1,"Pendrith, Taylor",0.075702,0,"McIlroy, Rory",2,0.036821,0.012171,0.961039,80
4,3,2025-02-09,1,"Thomas, Justin",0.034254,0,"Detry, Thomas",10,0.025017,0.012493,0.947368,132
5,7,2025-02-16,1,"Scheffler, Scottie",0.040417,0,"Aberg, Ludvig",19,0.014512,0.018471,0.584906,72
6,540,2025-02-23,1,"Bhatia, Akshay",0.033574,0,"Campbell, Brian",33,0.011532,0.012829,0.750000,132
7,10,2025-03-02,1,"Lowry, Shane",0.026670,0,"Highsmith, Joe",62,0.005410,0.014706,0.432836,144
8,9,2025-03-09,1,"Aberg, Ludvig",0.048776,0,"Henley, Russell",4,0.050026,0.018120,1.000000,72
9,11,2025-03-16,1,"Henley, Russell",0.049617,0,"McIlroy, Rory",11,0.009578,0.013774,0.605634,144


,year,event_id,event_start_date,dg_id,player_name,finish_num,is_winner,p_win,was_calibrated,sg_total_L40,sg_total_L24,sg_total_L12,pred_rank,p_win_raw,p_win_norm
0,2025,6,2025-01-12,14578,"Henley, Russell",10.0,0,0.054503,1,1.302175,1.322500,1.076083,1,0.054503,0.038401
1,2025,6,2025-01-12,25569,"Stevens, Sam",59.0,0,0.045750,1,0.463625,0.631708,-0.054167,2,0.045750,0.032234
2,2025,6,2025-01-12,18634,"McNealy, Maverick",45.0,0,0.041980,1,1.117150,0.975458,1.002417,3,0.041980,0.029577
3,2025,6,2025-01-12,27164,"Phillips, Chandler",NaN,0,0.039024,1,0.062000,0.265250,-0.800000,4,0.039024,0.027495
4,2025,6,2025-01-12,32330,"Clanton, Luke",NaN,0,0.037284,1,1.660250,1.660250,1.809500,5,0.037284,0.026268
5,2025,6,2025-01-12,23504,"Hoey, Rico",59.0,0,0.036378,1,0.573400,0.778958,0.355167,6,0.036378,0.025630
6,2025,6,2025-01-12,11276,"Horschel, Billy",NaN,0,0.034597,1,1.160025,1.018750,0.587833,7,0.034597,0.024376
7,2025,6,2025-01-12,13562,"Matsuyama, Hideki",16.0,0,0.033935,1,1.580925,1.627917,1.633583,8,0.033935,0.023910
8,2025,6,2025-01-12,10873,"Skinns, David",NaN,0,0.032796,1,0.267350,0.278083,-0.009583,9,0.032796,0.023106
9,2025,6,2025-01-12,25094,"Fishburn, Patrick",6.0,0,0.029673,1,0.037794,0.636792,1.677750,10,0.029673,0.020906


,p_bin,n,avg_pred_pwin,observed_win_rate
0,"(-0.001, 0.002]",343,0.001477,0.002915
1,"(0.002, 0.005]",1049,0.003698,0.001907
2,"(0.005, 0.01]",1549,0.006885,0.003873
3,"(0.01, 0.02]",725,0.013587,0.011034
4,"(0.02, 0.04]",178,0.026501,0.061798
5,"(0.04, 0.07]",23,0.049582,0.086957
6,"(0.07, 0.1]",3,0.080525,0.333333
7,"(0.1, 0.2]",0,NaN,NaN
8,"(0.2, 1.0]",0,NaN,NaN


,dg_id,player_name,starts,wins,avg_pwin,max_pwin
226,18417,"Scheffler, Scottie",18,5,0.056940,0.086220
63,10091,"McIlroy, Rory",14,3,0.039042,0.072526
204,17511,"Straka, Sepp",19,2,0.028281,0.052785
235,18628,"Campbell, Brian",22,2,0.010768,0.025895
139,14578,"Henley, Russell",16,1,0.035477,0.069113
127,14139,"Thomas, Justin",18,1,0.028365,0.053478
361,24968,"Griffin, Ben",26,1,0.026958,0.052003
205,17536,"Spaun, J.J.",22,1,0.022524,0.036852
417,27774,"Gotterup, Chris",21,1,0.022349,0.085797
119,13872,"Bradley, Keegan",19,1,0.022344,0.041906


In [22]:
# ============================================================
# WIN-ONLY MODEL (P(win)) — 2025 walk-forward, NO FUTURE INFO
# ============================================================
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score

# -----------------------------
# Feature set (SG-only)
# -----------------------------
WIN_FEATURES = ["sg_total_L40", "sg_total_L24", "sg_total_L12"]
USE_CALIBRATION = True
RANDOM_SEED = 42

def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def build_base_model():
    return HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=40,
        l2_regularization=0.0,
        random_state=RANDOM_SEED,
    )

def fit_model_with_year_sigmoid_calibration(train_df, feat_cols, label_col="y_win"):
    tr = train_df.dropna(subset=feat_cols + [label_col, "year"]).copy()
    Xtr = tr[feat_cols]
    ytr = tr[label_col].astype(int).values
    years = tr["year"].astype(int).values
    uniq_years = np.unique(years)

    base = build_base_model()

    # If not enough distinct years, skip calibration
    if (not USE_CALIBRATION) or (len(uniq_years) < 3):
        base.fit(Xtr, ytr)
        return base, None

    # LOYO OOF probs
    oof = np.full(len(tr), np.nan, dtype=float)
    for y in uniq_years:
        idx_te = np.where(years == y)[0]
        idx_tr = np.where(years != y)[0]
        if len(np.unique(ytr[idx_tr])) < 2:
            continue
        m = build_base_model()
        m.fit(Xtr.iloc[idx_tr], ytr[idx_tr])
        oof[idx_te] = m.predict_proba(Xtr.iloc[idx_te])[:, 1]

    ok = np.isfinite(oof)
    if ok.mean() < 0.8 or len(np.unique(ytr[ok])) < 2:
        base.fit(Xtr, ytr)
        return base, None

    # sigmoid calibrator on logit(oof)
    eps = 1e-6
    p = np.clip(oof[ok], eps, 1 - eps)
    logit_p = np.log(p / (1 - p)).reshape(-1, 1)

    cal = LogisticRegression(solver="lbfgs")
    cal.fit(logit_p, ytr[ok])

    base.fit(Xtr, ytr)
    return base, cal

def predict_proba_calibrated(base_model, calibrator, X):
    p = base_model.predict_proba(X)[:, 1]
    if calibrator is None:
        return p
    eps = 1e-6
    p2 = np.clip(p, eps, 1 - eps)
    logit_p = np.log(p2 / (1 - p2)).reshape(-1, 1)
    return calibrator.predict_proba(logit_p)[:, 1]

# ============================================================
# Names: schedule (optional) + event_skill fallback
# ============================================================
SCHED_2025 = None  # <-- set this if you want, e.g. "/Users/joshmacbook/python_projects/OAD/Data/in Use/OAD_2025.xlsx"

event_name_map = {}
if SCHED_2025 is not None:
    sched = pd.read_excel(SCHED_2025)
    sched["event_id"] = pd.to_numeric(sched.get("event_id"), errors="coerce").astype("Int64")
    if "event_name" in sched.columns:
        event_name_map = (
            sched.dropna(subset=["event_id"])
                 .drop_duplicates(subset=["event_id"])
                 .set_index("event_id")["event_name"]
                 .to_dict()
        )

# ============================================================
# event_skill + odds blend (stats-first)
# ============================================================
EVENT_SKILL_PATH = "/Data/in Use/event_skill.xlsx"
event_skill_df = pd.read_excel(EVENT_SKILL_PATH)
event_skill_df["year"] = pd.to_numeric(event_skill_df["year"], errors="coerce").astype("Int64")
event_skill_df["event_id"] = pd.to_numeric(event_skill_df["event_id"], errors="coerce").astype("Int64")
need_cols = [c for c in ["year","event_id","event_name","avg_skill","x_score"] if c in event_skill_df.columns]
event_skill_df = event_skill_df[need_cols].copy()

# if schedule wasn't provided, use event_skill names
if not event_name_map and "event_name" in event_skill_df.columns:
    event_name_map = (
        event_skill_df.dropna(subset=["event_id"])
                      .drop_duplicates(subset=["event_id"])
                      .set_index("event_id")["event_name"]
                      .to_dict()
    )

def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def _sigmoid(z):
    return 1 / (1 + np.exp(-z))

def blend_stats_odds_with_field_strength(board, year, ev_id, event_skill_df):
    """
    Requires:
      - p_win_raw  (stats model prob, not normalized)
      - close_odds (decimal odds)
    Adds:
      - p_win_blend_raw
      - p_win_blend_norm
      - blend_weight
      - x_score
    """
    b = board.copy()

    row = event_skill_df[(event_skill_df["year"] == year) & (event_skill_df["event_id"] == ev_id)]
    if row.empty or ("x_score" not in row.columns):
        x = np.nan
        w_event = 0.20
    else:
        x = float(row["x_score"].iloc[0])
        # map ~0.70..1.00 -> ~0.10..0.35, clamp
        w_event = 0.10 + 0.30 * ((x - 0.70) / (1.00 - 0.70))
        w_event = float(np.clip(w_event, 0.10, 0.35))

    # odds implied within-event
    p_odds_raw = 1.0 / pd.to_numeric(b["close_odds"], errors="coerce")
    p_odds_raw = p_odds_raw.fillna(p_odds_raw.median())
    p_odds = (p_odds_raw / (p_odds_raw.sum() + 1e-12)).values

    # stats within-event
    p_stats_raw = pd.to_numeric(b["p_win_raw"], errors="coerce").fillna(0.0)
    p_stats = (p_stats_raw / (p_stats_raw.sum() + 1e-12)).values

    z = _logit(p_stats) + w_event * _logit(p_odds)
    p_blend = _sigmoid(z)

    b["p_win_blend_raw"] = p_blend
    b["p_win_blend_norm"] = b["p_win_blend_raw"] / (b["p_win_blend_raw"].sum() + 1e-12)
    b["blend_weight"] = w_event
    b["x_score"] = x
    return b

# ============================================================
# 0) Prepare X and event order for 2025
# ============================================================
BACKTEST_YEAR = 2025
df_all = X.copy()

df_all["event_start_date"] = pd.to_datetime(df_all["event_start_date"], errors="coerce")

bt_events = (
    df_all[df_all["year"] == BACKTEST_YEAR][["event_id","event_start_date"]]
    .dropna()
    .drop_duplicates()
    .sort_values("event_start_date")
    .reset_index(drop=True)
)
print("2025 events found:", len(bt_events))

# ============================================================
# 1) Walk-forward
# ============================================================
event_boards = []
event_summ   = []
event_top10  = []

for _, ev in bt_events.iterrows():
    ev_id = int(ev["event_id"])
    ev_date = pd.to_datetime(ev["event_start_date"])
    ev_name = event_name_map.get(ev_id, "UNKNOWN_EVENT")
    print(f"\n=== Event {ev_id} | {ev_date.date()} | {ev_name} ===")

    train_df = df_all[
        (df_all["year"] < BACKTEST_YEAR) |
        ((df_all["year"] == BACKTEST_YEAR) & (df_all["event_start_date"] < ev_date))
    ].copy()

    test_df = df_all[(df_all["year"] == BACKTEST_YEAR) & (df_all["event_id"] == ev_id)].copy()
    if test_df.empty or train_df.empty:
        continue

    feat_cols = [c for c in WIN_FEATURES if c in train_df.columns]
    missing = set(WIN_FEATURES) - set(feat_cols)
    if missing:
        print("Missing features:", sorted(missing))
        continue

    base_model, calibrator = fit_model_with_year_sigmoid_calibration(train_df, feat_cols, label_col="y_win")
    p = predict_proba_calibrated(base_model, calibrator, test_df[feat_cols])
    was_cal = int(calibrator is not None)

    test_df = test_df.copy()
    test_df["p_win"] = p
    test_df["was_calibrated"] = was_cal
    test_df["is_winner"] = (pd.to_numeric(test_df["finish_num"], errors="coerce") == 1).astype(int)

    # Build board with features included
    base_cols = ["year","event_id","event_start_date","dg_id","player_name","finish_num","is_winner","p_win","was_calibrated"]
    board_cols = base_cols + feat_cols + (["close_odds"] if "close_odds" in test_df.columns else [])
    board = test_df[board_cols].copy()

    # Normalize stats-only win share
    board["p_win_raw"] = board["p_win"]
    board["p_win_norm"] = board["p_win_raw"] / (board["p_win_raw"].sum() + 1e-12)

    # Blend with odds if available
    if "close_odds" in board.columns:
        board = blend_stats_odds_with_field_strength(board, BACKTEST_YEAR, ev_id, event_skill_df)
        # rank by blended (you can switch to p_win_norm if you want stats-only ranking)
        board = board.sort_values("p_win_blend_norm", ascending=False).reset_index(drop=True)
    else:
        board = board.sort_values("p_win_norm", ascending=False).reset_index(drop=True)

    board["pred_rank"] = np.arange(1, len(board) + 1)
    board["event_name"] = ev_name

    # Append AFTER we’ve added p_win_* columns (this was one of your big issues)
    event_boards.append(board)

    # --- Top 10 display (robust selection) ---
    top10 = board.head(10).copy()

    show_cols = ["pred_rank","player_name","dg_id"]
    # show both stats-only and blended if present
    if "p_win_norm" in top10.columns:        show_cols += ["p_win_norm"]
    if "p_win_blend_norm" in top10.columns:  show_cols += ["p_win_blend_norm"]
    for c in ["sg_total_L40","sg_total_L24","sg_total_L12"]:
        if c in top10.columns:
            show_cols.append(c)

    if "p_win_norm" in top10.columns:
        top10["p_win_norm_pct"] = (100 * top10["p_win_norm"]).round(2)
    if "p_win_blend_norm" in top10.columns:
        top10["p_win_blend_norm_pct"] = (100 * top10["p_win_blend_norm"]).round(2)

    print(f"Top-10 ({ev_name})")
    display(top10[show_cols + [c for c in ["p_win_norm_pct","p_win_blend_norm_pct"] if c in top10.columns]])

    # store top10 rows for later concat
    top10_out = top10.copy()
    top10_out["event_date"] = ev_date
    event_top10.append(top10_out)

    # --- Event summary (use stats-only or blended consistently) ---
    winner_rows = board[board["is_winner"] == 1]
    winner_rank = int(winner_rows["pred_rank"].iloc[0]) if not winner_rows.empty else np.nan
    winner_name = winner_rows["player_name"].iloc[0] if not winner_rows.empty else None

    # score using stats-only normalized (stable baseline)
    eval_board = board.dropna(subset=["finish_num"]).copy()
    y_true = eval_board["is_winner"].values
    y_prob = eval_board["p_win_norm"].values
    brier = brier_score_loss(y_true, y_prob)
    auc = safe_auc(y_true, y_prob)

    top_pick = board.iloc[0]
    event_summ.append({
        "event_id": ev_id,
        "event_date": ev_date,
        "event_name": ev_name,
        "calibrated": was_cal,
        "top_pick": top_pick["player_name"],
        "top_pick_pwin": float(top_pick["p_win_norm"]),
        "top_pick_won": int(top_pick["is_winner"]),
        "winner_name": winner_name,
        "winner_rank": winner_rank,
        "brier": brier,
        "auc": auc,
        "n_field": len(board),
    })

boards_2025 = pd.concat(event_boards, ignore_index=True) if event_boards else pd.DataFrame()
summ_2025   = pd.DataFrame(event_summ).sort_values("event_date") if event_summ else pd.DataFrame()
top10_all_2025 = pd.concat(event_top10, ignore_index=True) if event_top10 else pd.DataFrame()

print("Events scored:", summ_2025["event_id"].nunique() if not summ_2025.empty else 0)
display(summ_2025.head(40))


2025 events found: 31

=== Event 6 | 2025-01-12 | Sony Open in Hawaii ===
Top-10 (Sony Open in Hawaii)


,pred_rank,player_name,dg_id,p_win_norm,p_win_blend_norm,sg_total_L40,sg_total_L24,sg_total_L12,p_win_norm_pct,p_win_blend_norm_pct
0,1,"Henley, Russell",14578,0.038401,0.064807,1.302175,1.322500,1.076083,3.84,6.48
1,2,"Matsuyama, Hideki",13562,0.023910,0.051539,1.580925,1.627917,1.633583,2.39,5.15
2,3,"McNealy, Maverick",18634,0.029577,0.046174,1.117150,0.975458,1.002417,2.96,4.62
3,4,"Clanton, Luke",32330,0.026268,0.038570,1.660250,1.660250,1.809500,2.63,3.86
4,5,"Stevens, Sam",25569,0.032234,0.031150,0.463625,0.631708,-0.054167,3.22,3.11
5,6,"Horschel, Billy",11276,0.024376,0.026524,1.160025,1.018750,0.587833,2.44,2.65
6,7,"Phillips, Chandler",27164,0.027495,0.026465,0.062000,0.265250,-0.800000,2.75,2.65
7,8,"Hoey, Rico",23504,0.025630,0.023941,0.573400,0.778958,0.355167,2.56,2.39
8,9,"Novak, Andrew",23475,0.019467,0.022696,0.737125,0.735833,0.055083,1.95,2.27
9,10,"McCarthy, Denny",19870,0.017747,0.021930,0.555325,0.663792,1.260583,1.77,2.19



=== Event 2 | 2025-01-19 | Desert Classic ===
Top-10 (Desert Classic)


,pred_rank,player_name,dg_id,p_win_norm,p_win_blend_norm,sg_total_L40,sg_total_L24,sg_total_L12,p_win_norm_pct,p_win_blend_norm_pct
0,1,"Meissner, Mac",28159,0.084884,0.102215,0.897600,0.632375,0.765000,8.49,10.22
1,2,"Greyserman, Max",23465,0.031604,0.046225,0.921875,1.453292,0.469167,3.16,4.62
2,3,"Hall, Harry",27194,0.025595,0.034135,0.793675,1.302875,1.564917,2.56,3.41
3,4,"Spaun, J.J.",17536,0.022765,0.027474,1.009425,2.055917,2.287333,2.28,2.75
4,5,"Smalley, Alex",18474,0.022152,0.022957,-0.479150,0.251500,1.336917,2.22,2.30
5,6,"Johnson, Zach",6986,0.024924,0.021269,0.253475,0.310750,0.370667,2.49,2.13
6,7,"Cantlay, Patrick",15466,0.012222,0.020613,1.320875,1.771667,1.368000,1.22,2.06
7,8,"Thomas, Justin",14139,0.010755,0.020494,0.790000,0.620542,0.034667,1.08,2.05
8,9,"Taylor, Nick",13126,0.018966,0.020332,-0.004425,0.263458,0.378333,1.90,2.03
9,10,"Burns, Sam",19483,0.011475,0.020050,1.267275,1.202792,1.510917,1.15,2.01



=== Event 4 | 2025-01-25 | Farmers Insurance Open ===
Top-10 (Farmers Insurance Open)


...

,pred_rank,player_name,dg_id,p_win_norm,p_win_blend_norm,sg_total_L40,sg_total_L24,sg_total_L12,p_win_norm_pct,p_win_blend_norm_pct
0,1,"Scheffler, Scottie",18417,0.064227,0.158301,3.234625,3.071958,3.356167,6.42,15.83
1,2,"Henley, Russell",14578,0.030113,0.033577,1.325825,1.152375,1.887917,3.01,3.36
2,3,"Griffin, Ben",24968,0.029521,0.031224,1.812100,1.610750,1.477833,2.95,3.12
3,4,"Hall, Harry",27194,0.034742,0.030706,1.568975,1.609417,1.390667,3.47,3.07
4,5,"Fleetwood, Tommy",12294,0.024250,0.029909,1.762450,1.861208,1.689500,2.42,2.99
5,6,"Fitzpatrick, Matt",17646,0.024165,0.027462,1.478325,1.996917,2.057333,2.42,2.75
6,7,"Aberg, Ludvig",23950,0.024556,0.027268,0.733675,1.322833,1.606167,2.46,2.73
7,8,"Gotterup, Chris",27774,0.025861,0.025038,1.513475,1.673417,1.459250,2.59,2.50
8,9,"English, Harris",14577,0.024768,0.023960,1.183275,1.321958,1.939500,2.48,2.40
9,10,"Rose, Justin",6093,0.029530,0.022625,0.499425,0.115875,2.218500,2.95,2.26



=== Event 28 | 2025-08-17 | BMW Championship ===
Top-10 (BMW Championship)


,pred_rank,player_name,dg_id,p_win_norm,p_win_blend_norm,sg_total_L40,sg_total_L24,sg_total_L12,p_win_norm_pct,p_win_blend_norm_pct
0,1,"Scheffler, Scottie",18417,0.080990,0.182346,3.397150,3.089167,3.733500,8.10,18.23
1,2,"McIlroy, Rory",10091,0.057376,0.088868,1.895850,1.594833,2.673250,5.74,8.89
2,3,"Fleetwood, Tommy",12294,0.052032,0.058314,1.809450,1.877333,2.400167,5.20,5.83
3,4,"Fitzpatrick, Matt",17646,0.048087,0.044613,1.568675,1.895042,0.961833,4.81,4.46
4,5,"Young, Cameron",26651,0.039416,0.040653,1.516500,1.519083,2.672167,3.94,4.07
5,6,"Gotterup, Chris",27774,0.041733,0.038532,1.388525,1.305208,0.030417,4.17,3.85
6,7,"Thomas, Justin",14139,0.030050,0.030785,0.982725,0.794000,0.733500,3.00,3.08
7,8,"Aberg, Ludvig",23950,0.026388,0.028141,0.894050,1.614750,2.066833,2.64,2.81
8,9,"Hall, Harry",27194,0.033653,0.026530,1.714475,1.853375,1.545167,3.37,2.65
9,10,"MacIntyre, Robert",23323,0.031720,0.025783,1.362925,1.437458,1.711833,3.17,2.58


Events scored: 31


,event_id,event_date,event_name,calibrated,top_pick,top_pick_pwin,top_pick_won,winner_name,winner_rank,brier,auc,n_field
0,6,2025-01-12,Sony Open in Hawaii,1,"Henley, Russell",0.038401,0,"Taylor, Nick",111,0.013240,0.106667,143
1,2,2025-01-19,Desert Classic,1,"Meissner, Mac",0.084884,0,"Straka, Sepp",18,0.013929,0.814286,156
2,4,2025-01-25,Farmers Insurance Open,1,"Griffin, Ben",0.036481,0,"English, Harris",13,0.013918,0.927536,154
3,5,2025-02-02,AT&T Pebble Beach Pro-Am,1,"Pendrith, Taylor",0.075702,0,"McIlroy, Rory",2,0.012171,0.961039,80
4,3,2025-02-09,Waste Management Phoenix Open,1,"Thomas, Justin",0.034254,0,"Detry, Thomas",10,0.012493,0.947368,132
5,7,2025-02-16,Genesis Open,1,"Scheffler, Scottie",0.040417,0,"Aberg, Ludvig",19,0.018471,0.584906,72
6,540,2025-02-23,Mexico Open at Vidanta,1,"Bhatia, Akshay",0.033574,0,"Campbell, Brian",33,0.012829,0.750000,132
7,10,2025-03-02,The Honda Classic,1,"Lowry, Shane",0.026670,0,"Highsmith, Joe",62,0.014706,0.432836,144
8,9,2025-03-09,Arnold Palmer Invitational presented by Master...,1,"Aberg, Ludvig",0.048776,0,"Henley, Russell",4,0.018120,1.000000,72
9,11,2025-03-16,THE PLAYERS Championship,1,"Henley, Russell",0.049617,0,"McIlroy, Rory",11,0.013774,0.605634,144
